# Phase 06A.07 — Statistical analysis and baseline-winner lock
Loads only complete 298-row artifacts, performs video-clustered paired inference, corrects multiple comparisons, and locks one reproducible baseline control.

In [ ]:
import json,os,sys
from pathlib import Path
PROJECT_ROOT=Path('/workspace/RoadBuddy'); SRC_DIR=PROJECT_ROOT/'src'
if str(SRC_DIR) not in sys.path: sys.path.insert(0,str(SRC_DIR))
os.chdir(PROJECT_ROOT)
from roadbuddy_common import *
from phase06a_common import *
seed_everything(SEED)

## Full-scope configuration and explicit artifact registry

In [ ]:
RUN_SCOPE='full'; CONTROL='Z-F1'; BOOTSTRAP_RESAMPLES=10_000; NONINFERIORITY_MARGIN=0.02
VALIDATION_IDS=PROJECT_ROOT/'data/splits/phase01/validation_sample_ids.json'; OUTPUT_DIR=PROJECT_ROOT/'outputs/phase06a/final_analysis'; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
prediction_paths={
 'Z-F1':PROJECT_ROOT/'outputs/phase06a/zero_shot/Z-F1/full/predictions.csv','Z-F3':PROJECT_ROOT/'outputs/phase06a/zero_shot/Z-F3/full/predictions.csv','Z-F8':PROJECT_ROOT/'outputs/phase06a/zero_shot/Z-F8/full/predictions.csv',
 'L16-F1':PROJECT_ROOT/'outputs/phase06a/lora_r16_evaluation/L16-F1/full/predictions.csv','L16-F3':PROJECT_ROOT/'outputs/phase06a/lora_r16_evaluation/L16-F3/full/predictions.csv','L16-F8':PROJECT_ROOT/'outputs/phase06a/lora_r16_evaluation/L16-F8/full/predictions.csv',
 'L8-F1':PROJECT_ROOT/'outputs/phase06a/rank_ablation/r8/full/predictions.csv','L32-F1':PROJECT_ROOT/'outputs/phase06a/rank_ablation/r32/full/predictions.csv','L64-F1':PROJECT_ROOT/'outputs/phase06a/rank_ablation/r64/full/predictions.csv'}
assert sha256_file(VALIDATION_IDS)==EXPECTED_VALIDATION_IDS_SHA256
frozen_ids=json.loads(VALIDATION_IDS.read_text(encoding='utf-8')); assert len(frozen_ids)==EXPECTED_VALIDATION_ROWS
missing={name:str(path) for name,path in prediction_paths.items() if not path.is_file()}; assert not missing,missing

## Integrity gates and leaderboard

In [ ]:
predictions={}; leaderboard=[]; per_class_rows=[]
for experiment,path in prediction_paths.items():
    frame=pd.read_csv(path); integrity=validate_prediction_artifact(frame,frozen_ids,run_scope='full'); metrics=compute_classification_metrics(frame); predictions[experiment]=frame
    distribution,metric_ci=group_cluster_metric_ci(frame,resamples=BOOTSTRAP_RESAMPLES,seed=SEED); distribution.to_csv(OUTPUT_DIR/f'bootstrap_metrics_{experiment}.csv',index=False)
    frames=int(experiment.split('-F')[-1]); rank=0 if experiment.startswith('Z') else int(experiment[1:].split('-')[0])
    leaderboard.append({'experiment':experiment,'frames':frames,'rank':rank,'accuracy':metrics['accuracy'],'accuracy_ci95_low':metric_ci['accuracy']['ci95_low'],'accuracy_ci95_high':metric_ci['accuracy']['ci95_high'],'macro_f1':metrics['macro_f1'],'macro_f1_ci95_low':metric_ci['macro_f1']['ci95_low'],'macro_f1_ci95_high':metric_ci['macro_f1']['ci95_high'],'parse_rate':metrics['parse_rate'],'parse_rate_ci95_low':metric_ci['parse_rate']['ci95_low'],'parse_rate_ci95_high':metric_ci['parse_rate']['ci95_high'],'mean_latency_seconds':float(frame.latency_seconds.mean()),'latency_ci95_low':metric_ci['mean_latency_seconds']['ci95_low'],'latency_ci95_high':metric_ci['mean_latency_seconds']['ci95_high'],'mean_realized_tiles':float(frame.realized_tile_count.mean()),'tiles_ci95_low':metric_ci['mean_realized_tiles']['ci95_low'],'tiles_ci95_high':metric_ci['mean_realized_tiles']['ci95_high']})
    for label,values in metrics['per_class'].items(): per_class_rows.append({'experiment':experiment,'class':label,**values,'f1_ci95_low':metric_ci[f'class_f1_{label}']['ci95_low'],'f1_ci95_high':metric_ci[f'class_f1_{label}']['ci95_high']})
    save_json(OUTPUT_DIR/f'confusion_matrix_{experiment}.json',metrics['confusion_matrix'])
leaderboard=pd.DataFrame(leaderboard).sort_values(['accuracy','macro_f1'],ascending=False).reset_index(drop=True); leaderboard.to_csv(OUTPUT_DIR/'final_leaderboard.csv',index=False)
pd.DataFrame(per_class_rows).to_csv(OUTPUT_DIR/'per_class_metrics.csv',index=False)

## Paired group-cluster bootstrap, McNemar, and Holm correction

In [ ]:
control=predictions[CONTROL]; comparison_rows=[]; raw_p={}; transitions=[]
for experiment,frame in predictions.items():
    if experiment==CONTROL: continue
    distribution,bootstrap=paired_group_cluster_bootstrap(control,frame,resamples=BOOTSTRAP_RESAMPLES,seed=SEED); distribution.to_csv(OUTPUT_DIR/f'bootstrap_{experiment}_vs_{CONTROL}.csv',index=False)
    paired=control[['sample_id','correct']].merge(frame[['sample_id','correct']],on='sample_id',suffixes=('_control','_candidate'),validate='one_to_one')
    paired['experiment']=experiment; paired['transition']=np.select([~paired.correct_control & paired.correct_candidate,paired.correct_control & ~paired.correct_candidate],['wrong_to_right','right_to_wrong'],default='unchanged'); transitions.append(paired)
    mcnemar=exact_mcnemar(paired.correct_control,paired.correct_candidate); raw_p[experiment]=mcnemar['exact_two_sided_p']
    comparison_rows.append({'experiment':experiment,'control':CONTROL,**{f'bootstrap_{metric}_{key}':value for metric,items in bootstrap.items() for key,value in items.items()},**mcnemar})
adjusted=holm_adjust(raw_p)
for row in comparison_rows: row['holm_adjusted_p']=adjusted[row['experiment']]
comparisons=pd.DataFrame(comparison_rows); comparisons.to_csv(OUTPUT_DIR/'paired_statistical_tests.csv',index=False)
pd.concat(transitions,ignore_index=True).to_csv(OUTPUT_DIR/'paired_transitions.csv',index=False)

## Preregistered non-inferiority and deterministic winner rule

In [ ]:
best_name=leaderboard.iloc[0].experiment; best_frame=predictions[best_name]; eligible=[]; noninferiority=[]
for experiment,frame in predictions.items():
    if experiment==best_name:
        low=high=0.0; is_noninferior=True
    else:
        _,summary=paired_group_cluster_bootstrap(best_frame,frame,resamples=BOOTSTRAP_RESAMPLES,seed=SEED+1); low=summary['accuracy_delta']['ci95_low']; high=summary['accuracy_delta']['ci95_high']; is_noninferior=low>=-NONINFERIORITY_MARGIN
    noninferiority.append({'experiment':experiment,'reference_best':best_name,'accuracy_delta_ci95_low':low,'accuracy_delta_ci95_high':high,'margin':NONINFERIORITY_MARGIN,'noninferior':is_noninferior})
    if is_noninferior: eligible.append(experiment)
complexity=leaderboard.set_index('experiment').copy(); complexity['fine_tuned']=~complexity.index.to_series().str.startswith('Z')
winner_table=complexity.loc[eligible].reset_index().sort_values(['macro_f1','frames','fine_tuned','rank','mean_latency_seconds'],ascending=[False,True,True,True,True])
winner=winner_table.iloc[0].experiment
pd.DataFrame(noninferiority).to_csv(OUTPUT_DIR/'noninferiority_analysis.csv',index=False)
runtime_paths={name:(PROJECT_ROOT/('outputs/phase06a/zero_shot' if name.startswith('Z-') else 'outputs/phase06a/lora_r16_evaluation')/name/'full/runtime.json') for name in ['Z-F1','Z-F3','Z-F8','L16-F1','L16-F3','L16-F8']}
rank_resources=pd.read_csv(PROJECT_ROOT/'outputs/phase06a/rank_ablation/rank_summary_full.csv').set_index('rank')
r16_training=json.loads((PROJECT_ROOT/'outputs/phase06a/lora_r16_training/full/stage_b_final_retrain/training_result.json').read_text(encoding='utf-8'))
resource_rows=[]
for experiment in leaderboard.experiment:
    if experiment in runtime_paths:
        resource=json.loads(runtime_paths[experiment].read_text(encoding='utf-8')); peak_vram=resource.get('peak_vram_gib'); peak_vram_scope='inference'
    else:
        rank=int(experiment[1:].split('-')[0]); peak_vram=r16_training.get('peak_vram_gib') if rank==16 else rank_resources.loc[rank,'peak_vram_gib']; peak_vram_scope='training'
    resource_rows.append({'experiment':experiment,'peak_vram_gib':peak_vram,'peak_vram_scope':peak_vram_scope})
efficiency=leaderboard[['experiment','frames','rank','mean_latency_seconds','mean_realized_tiles']].merge(pd.DataFrame(resource_rows),on='experiment',validate='one_to_one'); efficiency.to_csv(OUTPUT_DIR/'efficiency_table.csv',index=False)

## Lock winner and reproducibility report

In [ ]:
winner_path=prediction_paths[winner]; winner_manifest={'phase':'06A','status':'complete','winner':winner,'selection_rule':'highest accuracy; if no clear superiority choose least-complex configuration within preregistered 0.02 non-inferiority margin','primary_metric':'accuracy','secondary_metric':'macro_f1','validation_rows':EXPECTED_VALIDATION_ROWS,'validation_ids_sha256':sha256_file(VALIDATION_IDS),'winner_predictions':artifact_record(winner_path),'source_experiments':{name:artifact_record(path) for name,path in prediction_paths.items()},'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'mcnemar_role':'secondary; sample-pair test does not remove within-video correlation','holm_corrected':True}
save_json(OUTPUT_DIR/'baseline_winner_manifest.json',winner_manifest)
report=f'''# RoadBuddy Phase 06A reproducibility report

Status: complete

Baseline winner: **{winner}**

Validation membership: 298 frozen samples; SHA-256 `{winner_manifest['validation_ids_sha256']}`.

Primary inference uses paired video-group cluster bootstrap with {BOOTSTRAP_RESAMPLES:,} resamples. Exact McNemar is secondary and Holm-adjusted across comparisons. Absence of significance is not interpreted as model equivalence.

No held-out-test artifact is created by Phase 06A.
'''
(OUTPUT_DIR/'PHASE06A_REPRODUCIBILITY_REPORT.md').write_text(report,encoding='utf-8')
save_json(OUTPUT_DIR/'PHASE06A_FINAL_STATUS.json',{'phase':'06A','status':'complete','winner':winner,'validation_rows':EXPECTED_VALIDATION_ROWS,'experiments':len(predictions)})
display(leaderboard); display(comparisons); display(winner_manifest)

## Interpretation constraint
The winner is a validation-selected control, not an unbiased held-out-test estimate. Temporal Grounding may begin only after this manifest is locked.